# Fire catalog: crawl the NIFC databases, gate the incidents, write `catalog/fires.csv`

The raw data grab (`kaggle_data_grab.ipynb`) fetches every channel for the fires listed in its target repo's
`catalog/fires.csv`. This notebook makes that list from the source databases, with no data from any built dataset:

1. **Scan** crawls every record of every NIFC wildfire perimeter archive, all fields as served, and the WFIGS incident
   record of every IRWIN id they name:
   - GeoMAC `Historic_Geomac_Perimeters_<year>` (2014-2019)
   - NIFS `National_Incident_Feature_Service_2018`, `2019_NIFS_OpenData`
   - Operational Data Archive 2020-2021 (geodatabase downloads) and `Operational_Data_Archive_2022`...`2025`
   - `WFIGS_Incident_Locations` (97 fields: discovery time, cause, incident type, final acres, containment, fuel,
     complex, point of origin)

   Each table goes to the repo untouched as `catalog/raw/perimeters/<year>_<archive>_<layer>.parquet` and
   `catalog/raw/incidents/wfigs_<year>.parquet`. A table already in the repo is not crawled again, so a stopped
   session just resumes.
2. **Build** applies the gates and writes:
   - `catalog/fires.csv`: one row per fire
   - `catalog/incidents.csv`: every incident, with its gate values and the reason it is not a fire
   - `catalog/gates.json`: the settings and counts

   Records are never edited. The gates read each archive by its own rules: GeoMAC times are local clock times
   stored as UTC; NIFS / ODA times are true UTC, and their versions and `DeleteThis` flags are honoured.

| gate | default | how it is read |
|---|---|---|
| years | 2014-2025 | year of the last perimeter of the fire's main run |
| region | continental US | record centroid inside (-125, 24, -66, 50) |
| wildfire | yes | NIFS `FeatureCategory` wildfire perimeter; WFIGS `IncidentTypeCategory` not RX |
| size | >= 1,000 acres | largest `GISAcres` of a counted perimeter |
| days | >= 3 burn days | local solar days (before 06:00 counts for the day before) with a counted perimeter |
| counted | | hand sketches and modeled shapes do not count for size or days |
| main run | | records not chained within 25 km, or more than 10 days apart, split the fire; the run with the most days sets window and box |

Fire ids are `"F" + sha1(incident key)[:10]`, the key being the IRWIN id (else name and unique fire id), so a fire
keeps the id, and the folders on the Hub, it had in the earlier catalog.

**Before you run**
- Settings → Internet: **on**
- Add-ons → Secrets: `HF_TOKEN_ALL`, a Hugging Face token with write access (label set in `SECRET_LABELS`).
  With `UPLOAD = False`, no token is needed and everything stays under `WORK`.

In [ ]:
# ---- settings
HF_REPO = "your-name/wildfire-raw-2014-2025"   # target dataset repo (created if missing); the data grab reads it
PRIVATE = True                                 # only used when the repo is created
UPLOAD = True                                  # False: crawl and build locally only (nothing sent to the Hub)
WORK = "/tmp/raw"                              # scratch; the local copy of the catalog is WORK/catalog_repo
YEARS = "2014-2025"                            # years crawled and kept
MIN_ACRES = 1000                               # gate: largest mapped size
MIN_DAYS = 3                                   # gate: burn days with a mapped perimeter
THREADS = 16

# earlier catalog to compare with (the built dataset's); set to None to skip the comparison
OLD_CATALOG = ("Javyn123/wildfire-spread-2014-2025", "catalog/fires.csv")

SECRET_LABELS = {"HF_TOKEN": "HF_TOKEN_ALL"}   # left: what the script reads; right: the Kaggle secret's label
BRANCH = "data_grab_satfire"                   # WildfireRepo1 branch the code comes from

In [ ]:
# ---- the project code, BRANCH (settings cell)
!git clone -q --depth 1 -b {BRANCH} https://github.com/GT-Fire-Research-and-Innovation/WildfireRepo1.git 2>/dev/null || (cd WildfireRepo1 && git fetch -q --depth 1 origin {BRANCH} && git checkout -q -B {BRANCH} FETCH_HEAD)
!cd WildfireRepo1 && git log -1 --format='%h %s'

In [ ]:
# ---- packages: pyogrio reads the 2020-2021 geodatabases, pyarrow the Parquet tables
!pip install -q -U pyogrio pyarrow shapely "huggingface_hub>=0.24"

In [ ]:
# ---- secrets -> the environment variables the script reads (values never printed)
import os
try:
    from kaggle_secrets import UserSecretsClient
    _s = UserSecretsClient()
    for _env, _label in SECRET_LABELS.items():
        try:
            os.environ[_env] = _s.get_secret(_label)
        except Exception as _e:
            print(f"Kaggle secret {_label!r} not readable ({type(_e).__name__}): add it under Add-ons > Secrets "
                  f"with exactly this label and switch it on for this notebook (or change SECRET_LABELS)")
except ImportError:
    pass                                       # not on Kaggle: use the existing environment
print({env: ("set" if os.environ.get(env) else "MISSING") for env in SECRET_LABELS})

In [ ]:
import subprocess, sys
from pathlib import Path

REPO_DIR = "WildfireRepo1"
CATALOG = Path(WORK) / "catalog_repo" / "catalog"   # fires.csv, incidents.csv, gates.json, raw/

def _run(args):
    p = subprocess.Popen([sys.executable, "-u", "-m", "fah", *args], cwd=REPO_DIR, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    return p.wait()

def catalog(step):
    """step: scan | build | all"""
    args = ["--catalog", step, "--work", WORK, "--years", YEARS, "--min-acres", str(MIN_ACRES),
            "--min-days", str(MIN_DAYS), "--threads", str(THREADS)]
    args += ["--hf-repo", HF_REPO] + (["--private"] if PRIVATE else []) if UPLOAD else ["--dry-run"]
    return _run(args)

if UPLOAD:   # stop here unless the token can write to HF_REPO (prints exactly what to fix)
    assert _run(["--check", "--hf-repo", HF_REPO]) == 0, "fix the setup above, then rerun from the secrets cell"

## 1. Scan: every perimeter record and every WFIGS incident, untouched

In [ ]:
assert catalog("scan") == 0, "the crawl stopped (see above); run this cell again: finished tables are kept"

## 2. Build: the gates, then `fires.csv`

In [ ]:
assert catalog("build") == 0

## 3. What came out

Everything below reads the local copy under `WORK/catalog_repo/catalog` (the same files that were uploaded).

In [ ]:
import json
import pandas as pd
pd.set_option("display.width", 200, "display.max_columns", 50, "display.max_rows", 200)

fires = pd.read_csv(CATALOG / "fires.csv")
incidents = pd.read_csv(CATALOG / "incidents.csv", low_memory=False)
gates = json.loads((CATALOG / "gates.json").read_text())
print(json.dumps(gates["gates"]), "\n")
print(f"{len(fires)} fires from {len(incidents)} incidents\n")
print("fires per year and archive (both = mapped in GeoMAC and NIFS):")
display(pd.crosstab(fires["year"], fires["archive"], margins=True))
print("incidents that are not fires, by the first gate they fail:")
display(incidents.loc[incidents["reason"].fillna("") != "", "reason"].value_counts().to_frame("incidents"))

In [ ]:
# ---- every variable crawled: the fields of each raw table, which tables have them, and how often they are filled
import pyarrow.parquet as pq

def inventory(kind):
    rows = []
    for path in sorted((CATALOG / "raw" / kind).glob("*.parquet")):
        t = pq.read_table(path)
        for name in t.column_names:
            col = t.column(name)
            rows.append({"field": name, "table": path.stem, "rows": t.num_rows,
                         "filled": t.num_rows - col.null_count})
    df = pd.DataFrame(rows)
    out = df.groupby("field").agg(tables=("table", "nunique"), rows=("rows", "sum"), filled=("filled", "sum"))
    out["filled_share"] = (out["filled"] / out["rows"]).round(3)
    out["in"] = df.groupby("field")["table"].agg(lambda s: ", ".join(sorted(s))[:90])
    return out.sort_values(["tables", "field"], ascending=[False, True])

print("perimeter records: every field of every archive layer (all kept in catalog/raw/perimeters)")
display(inventory("perimeters"))
print("WFIGS incidents: every field (all kept in catalog/raw/incidents)")
display(inventory("incidents"))

In [ ]:
# ---- fires.csv: every column, how often it is filled, and the Camp Fire's row as an example
camp = fires[fires["fire_id"] == "F2003403265"]
cols = pd.DataFrame({"dtype": fires.dtypes.astype(str), "filled_share": fires.notna().mean().round(3),
                     "Camp (F2003403265)": camp.iloc[0] if len(camp) else None})
display(cols)

In [ ]:
# ---- against the earlier catalog: same ids, new ones, and why the missing ones are not fires now
if OLD_CATALOG:
    from huggingface_hub import hf_hub_download
    try:
        old = pd.read_csv(hf_hub_download(OLD_CATALOG[0], OLD_CATALOG[1], repo_type="dataset"))
    except Exception as e:
        old = None
        print(f"earlier catalog not readable ({type(e).__name__}): comparison skipped")
    if old is not None:
        y0, y1 = (int(y) for y in YEARS.split("-"))
        old = old[old["year"].between(y0, y1)]
        same = set(old["fire_id"]) & set(fires["fire_id"])
        new, gone = set(fires["fire_id"]) - same, set(old["fire_id"]) - same
        print(f"earlier: {len(old)} fires, now: {len(fires)}; in both: {len(same)}, new: {len(new)}, "
              f"missing: {len(gone)}")
        why = incidents.set_index("fire_id")["reason"].reindex(sorted(gone)).fillna("not in the crawled archives")
        display(why.value_counts().to_frame("earlier fires no longer in"))
        both = fires.set_index("fire_id").loc[sorted(same)]
        prev = old.set_index("fire_id").loc[sorted(same)]
        d = (both[["west", "south", "east", "north"]] - prev[["west", "south", "east", "north"]]).abs().max(axis=1)
        print(f"box change of fires in both (degrees): median {d.median():.4f}, 95th pct {d.quantile(0.95):.4f}")